# 05. KR-SBERT 기술 임베딩 생성

기술 문서 문장을 한국어 문장 유사도 모델로 벡터화합니다. 결과 벡터와 기술 메타데이터는 원본 데이터의 재식별 가능성을 고려해 공개하지 않습니다.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

INPUT_PATH = Path('../artifacts/technology_embedding_input.csv')
EMBEDDING_OUTPUT_PATH = Path('../artifacts/technology_embeddings.npy')
METADATA_OUTPUT_PATH = Path('../artifacts/technology_metadata.csv')
TEXT_COLUMN = 'embedding_text'
METADATA_COLUMNS = ['provider_company', 'technology_type', 'technology_name', 'description']
MODEL_NAME = 'snunlp/KR-SBERT-V40K-klueNLI-augSTS'

if not INPUT_PATH.exists():
    raise FileNotFoundError(f'Private input is not available: {INPUT_PATH}')

technology_df = pd.read_csv(INPUT_PATH)
missing_columns = set(METADATA_COLUMNS + [TEXT_COLUMN]) - set(technology_df.columns)
if missing_columns:
    raise KeyError(f'Missing required columns: {sorted(missing_columns)}')

texts = technology_df[TEXT_COLUMN].fillna('').astype(str).str.strip()
if texts.eq('').any():
    raise ValueError('Embedding input contains empty technology text.')

EMBEDDING_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
model = SentenceTransformer(MODEL_NAME)
embeddings = model.encode(texts.tolist(), show_progress_bar=True)
np.save(EMBEDDING_OUTPUT_PATH, embeddings)
technology_df.loc[:, METADATA_COLUMNS].to_csv(
    METADATA_OUTPUT_PATH,
    index=False,
    encoding='utf-8-sig',
)

print(f'Saved {embeddings.shape[0]} embeddings with dimension {embeddings.shape[1]}.')